In [1]:
from pathlib import Path

OUTPUT_DIR = Path("notebook3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Created:", OUTPUT_DIR)

Created: notebook3_outputs


In [2]:
import shutil
from pathlib import Path

for f in Path(".").glob("results_batch_*.csv"):
    shutil.move(str(f), OUTPUT_DIR / f.name)

print("Moved batch CSVs into", OUTPUT_DIR)

Moved batch CSVs into notebook3_outputs


In [3]:
batch_csvs = sorted(OUTPUT_DIR.glob("results_batch_*.csv"))

print("Count:", len(batch_csvs))
print("First 5:", [f.name for f in batch_csvs[:5]])
print("Last 5:", [f.name for f in batch_csvs[-5:]])

Count: 53
First 5: ['results_batch_01.csv', 'results_batch_02.csv', 'results_batch_03.csv', 'results_batch_04.csv', 'results_batch_05.csv']
Last 5: ['results_batch_49.csv', 'results_batch_50.csv', 'results_batch_51.csv', 'results_batch_52.csv', 'results_batch_53.csv']


In [4]:
import pandas as pd

batch_csvs = sorted(OUTPUT_DIR.glob("results_batch_*.csv"))

if not batch_csvs:
    raise ValueError("No completed batch result CSVs found.")

merged_labels_df = pd.concat(
    [pd.read_csv(f) for f in batch_csvs],
    ignore_index=True
)

merged_labels_df = (
    merged_labels_df
    .sort_values("community_id")
    .drop_duplicates(subset=["community_id"], keep="last")
    .reset_index(drop=True)
)

merged_labels_df.to_csv(OUTPUT_DIR / "all_community_labels_merged.csv", index=False)

print("Saved: all_community_labels_merged.csv")
print("Merged shape:", merged_labels_df.shape)
display(merged_labels_df.head(10))

Saved: all_community_labels_merged.csv
Merged shape: (1307, 16)


,community_id,community_size,dominant_experience_level,entry_accessibility,community_label,defining_skills,defining_responsibility_signals,confidence,level_agreement,access_agreement,label_agreement,overall_agreement,calls_used,top_title_keywords,raw_call_outputs,batch_id
0,0,1858,mid,restricted_entry_point,Nursing Professionals,"['LPN certification', 'medical care', 'patient...","['provide patient care', 'assist registered nu...",high,1.000,1.000,0.333,0.778,3,"nurse, registered, care, licensed, lpn, practi...","[{""community_id"": 0, ""dominant_experience_leve...",1
1,1,1308,mid,moderate_entry_barrier,Sales & Marketing,"['sales', 'sales strategy', 'customer relation...","['develop and execute sales strategy', 'mainta...",medium,1.000,1.000,0.667,0.889,3,"sales, manager, representative, salesperson, m...","[{""community_id"": 1, ""dominant_experience_leve...",1
2,2,1289,mid,moderate_entry_barrier,Accounting Professionals,"['accounting', 'financial', 'payable']","['monthly close process', 'cross functional te...",medium,1.000,1.000,0.667,0.889,3,"accountant, accounting, senior, manager, staff...","[{""community_id"": 2, ""dominant_experience_leve...",1
3,3,857,mid,moderate_entry_barrier,Retail Management,"['team management', 'customer service', 'sales...","['motivating team members', 'meeting sales goa...",medium,1.000,1.000,0.667,0.889,3,"manager, store, assistant, sales, supervisor, ...","[{""community_id"": 3, ""dominant_experience_leve...",1
4,4,787,mid,moderate_entry_barrier,Construction Project Management,"['project management', 'construction', 'estima...","['manage projects', 'align with company polici...",high,1.000,1.000,0.667,0.889,3,"project, manager, construction, senior, engine...","[{""community_id"": 4, ""dominant_experience_leve...",1
5,5,684,junior,clear_entry_point,Warehouse Operations,"['forklift', 'computer skills - sap', 'materia...","['select and operate equipment', 'move and shi...",high,1.000,1.000,1.000,1.000,3,"warehouse, driver, shift, store, operator, del...","[{""community_id"": 5, ""dominant_experience_leve...",1
6,6,611,junior,clear_entry_point,Maintenance Technicians,"['repair', 'maintenance', 'install']","['installing repairing and maintaining', 'star...",high,1.000,1.000,0.667,0.889,3,"technician, maintenance, service, mechanic, sh...","[{""community_id"": 6, ""dominant_experience_leve...",1
7,7,553,mid,moderate_entry_barrier,Data Analytics Professionals,"['data engineering', 'business analytics', 'da...","['collaborate with specialized professionals',...",medium,1.000,1.000,0.333,0.778,3,"data, engineer, analyst, senior, business, ana...","[{""community_id"": 7, ""dominant_experience_leve...",1
8,8,482,mid,moderate_entry_barrier,Sales & Business Development,"['consultative sales', 'customer adoption', 'g...","['drive customer growth', 'onboard strategic c...",high,0.667,0.667,0.667,0.667,3,"account, executive, manager, sales, senior, bu...","[{""community_id"": 8, ""dominant_experience_leve...",1
9,9,472,mid,moderate_entry_barrier,Banking Professionals,"['relationship management', 'client service', ...","['client interaction', 'sales and service targ...",medium,1.000,1.000,1.000,1.000,3,"banker, personal, teller, relationship, client...","[{""community_id"": 9, ""dominant_experience_leve...",1


In [5]:
import ast

for col in ["defining_skills", "defining_responsibility_signals"]:
    if col in merged_labels_df.columns:
        merged_labels_df[col] = merged_labels_df[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x
        )

print("Parsed list-like columns.")

Parsed list-like columns.


In [6]:
community_summary_df = pd.read_csv("community_summary.csv", engine="python", on_bad_lines="skip")

stage3_df = community_summary_df.merge(
    merged_labels_df,
    on="community_id",
    how="inner",
    suffixes=("_nb2", "_nb3")
)

print("Merged stage3 dataframe shape:", stage3_df.shape)
display(stage3_df.head(5))

Merged stage3 dataframe shape: (1307, 57)


,community_id,community_size_nb2,dominant_experience_reference,reference_share_junior,reference_share_mid,reference_share_senior,top_title_keywords_nb2,rep_job_id_1,rep_text_1,rep_similarity_1,...,defining_responsibility_signals,confidence,level_agreement,access_agreement,label_agreement,overall_agreement,calls_used,top_title_keywords_nb3,raw_call_outputs,batch_id
0,0,1858,mid,0.298,0.686,0.017,"nurse, registered, care, licensed, lpn, practi...",3884831783,rn registered nurse agency free facility [SEP]...,0.9132,...,"[provide patient care, assist registered nurses]",high,1.0,1.0,0.333,0.778,3,"nurse, registered, care, licensed, lpn, practi...","[{""community_id"": 0, ""dominant_experience_leve...",1
1,1,1308,junior,0.492,0.440,0.068,"sales, manager, representative, salesperson, m...",3902790352,sales representative - packaging [SEP] job pur...,0.9019,...,"[develop and execute sales strategy, maintain ...",medium,1.0,1.0,0.667,0.889,3,"sales, manager, representative, salesperson, m...","[{""community_id"": 1, ""dominant_experience_leve...",1
2,2,1289,mid,0.200,0.711,0.088,"accountant, accounting, senior, manager, staff...",3906256333,staff accountant [SEP] position overview we ar...,0.9147,...,"[monthly close process, cross functional team ...",medium,1.0,1.0,0.667,0.889,3,"accountant, accounting, senior, manager, staff...","[{""community_id"": 2, ""dominant_experience_leve...",1
3,3,857,mid,0.448,0.543,0.009,"manager, store, assistant, sales, supervisor, ...",3904988025,assistant store manager [SEP] what s the role ...,0.8587,...,"[motivating team members, meeting sales goals]",medium,1.0,1.0,0.667,0.889,3,"manager, store, assistant, sales, supervisor, ...","[{""community_id"": 3, ""dominant_experience_leve...",1
4,4,787,mid,0.166,0.785,0.048,"project, manager, construction, senior, engine...",3903466941,project manager [SEP] position summarythis pos...,0.9135,...,"[manage projects, align with company policies]",high,1.0,1.0,0.667,0.889,3,"project, manager, construction, senior, engine...","[{""community_id"": 4, ""dominant_experience_leve...",1


In [7]:
def get_size(row):
    if "community_size" in row.index:
        return int(row["community_size"])
    if "community_size_nb2" in row.index:
        return int(row["community_size_nb2"])
    if "community_size_nb3" in row.index:
        return int(row["community_size_nb3"])
    return 0

def build_analyst_interpretation(row):
    level = row["dominant_experience_level"]
    access = row["entry_accessibility"]
    label = row["community_label"]
    agreement = float(row["overall_agreement"])

    if access == "clear_entry_point" and level == "junior":
        return f"{label} is a high-value entry-point community with relatively accessible labor-market conditions."
    if access == "restricted_entry_point" and level == "junior":
        return f"{label} is junior-framed but behaves like a restricted-entry segment that may discourage genuine entry candidates."
    if access == "restricted_entry_point":
        return f"{label} is a restricted-entry community with elevated barriers and should be monitored as a high-selectivity segment."
    if agreement < 0.67:
        return f"{label} is semantically ambiguous and should be reviewed manually before relying on it."
    return f"{label} is a moderate-barrier community representing a stable labor-market segment."

def build_recruiter_action(row):
    access = row["entry_accessibility"]
    level = row["dominant_experience_level"]
    skills = ", ".join(row["defining_skills"]) if isinstance(row["defining_skills"], list) else str(row["defining_skills"])

    if access == "clear_entry_point" and level == "junior":
        return f"Prioritize this community for entry-level sourcing and screen for {skills}."
    if access == "restricted_entry_point" and level == "junior":
        return "Audit whether the role mixes junior framing with advanced requirements."
    if access == "restricted_entry_point":
        return "Use this community as a benchmark for selective hiring and communicate barriers clearly."
    return f"Use this community to refine hiring criteria around {skills}."

def build_candidate_advice(row):
    skills = ", ".join(row["defining_skills"]) if isinstance(row["defining_skills"], list) else str(row["defining_skills"])
    responsibilities = ", ".join(row["defining_responsibility_signals"]) if isinstance(row["defining_responsibility_signals"], list) else str(row["defining_responsibility_signals"])
    access = row["entry_accessibility"]

    if access == "clear_entry_point":
        return f"This is a realistic entry route. Show evidence of {skills} and {responsibilities}."
    if access == "restricted_entry_point":
        return f"This is not an easy entry route. Strengthen proof of {skills} before targeting it."
    return f"This segment is moderately accessible. Improve evidence of {skills} and {responsibilities}."

def build_policy_signal(row):
    access = row["entry_accessibility"]
    size = get_size(row)

    if access == "clear_entry_point" and size >= 100:
        return "Potential workforce-entry channel worth monitoring for access and transition programs."
    if access == "restricted_entry_point" and size >= 100:
        return "Potential barrier-heavy segment worth monitoring for qualification inflation."
    return "Monitor as a secondary labor-market segment."

def build_review_flag(row):
    if float(row["overall_agreement"]) < 0.67:
        return "manual_review_recommended"
    if row["confidence"] == "low":
        return "manual_review_recommended"
    return "ok"

stage3_df["analyst_interpretation"] = stage3_df.apply(build_analyst_interpretation, axis=1)
stage3_df["recruiter_action"] = stage3_df.apply(build_recruiter_action, axis=1)
stage3_df["candidate_advice"] = stage3_df.apply(build_candidate_advice, axis=1)
stage3_df["policy_signal"] = stage3_df.apply(build_policy_signal, axis=1)
stage3_df["review_flag"] = stage3_df.apply(build_review_flag, axis=1)

display(stage3_df[[
    "community_id",
    "community_label",
    "dominant_experience_level",
    "entry_accessibility",
    "overall_agreement",
    "review_flag"
]].head(10))

,community_id,community_label,dominant_experience_level,entry_accessibility,overall_agreement,review_flag
0,0,Nursing Professionals,mid,restricted_entry_point,0.778,ok
1,1,Sales & Marketing,mid,moderate_entry_barrier,0.889,ok
2,2,Accounting Professionals,mid,moderate_entry_barrier,0.889,ok
3,3,Retail Management,mid,moderate_entry_barrier,0.889,ok
4,4,Construction Project Management,mid,moderate_entry_barrier,0.889,ok
5,5,Warehouse Operations,junior,clear_entry_point,1.000,ok
6,6,Maintenance Technicians,junior,clear_entry_point,0.889,ok
7,7,Data Analytics Professionals,mid,moderate_entry_barrier,0.778,ok
8,8,Sales & Business Development,mid,moderate_entry_barrier,0.667,manual_review_recommended
9,9,Banking Professionals,mid,moderate_entry_barrier,1.000,ok


In [8]:
size_col = "community_size"
if "community_size" not in stage3_df.columns:
    if "community_size_nb2" in stage3_df.columns:
        size_col = "community_size_nb2"
    elif "community_size_nb3" in stage3_df.columns:
        size_col = "community_size_nb3"

entry_df = stage3_df[
    (stage3_df["entry_accessibility"] == "clear_entry_point") &
    (stage3_df["dominant_experience_level"] == "junior")
].copy().sort_values(
    by=["overall_agreement", size_col],
    ascending=[False, False]
).reset_index(drop=True)

restricted_df = stage3_df[
    stage3_df["entry_accessibility"] == "restricted_entry_point"
].copy().sort_values(
    by=["overall_agreement", size_col],
    ascending=[False, False]
).reset_index(drop=True)

ambiguous_df = stage3_df[
    stage3_df["review_flag"] == "manual_review_recommended"
].copy().sort_values(
    by=["overall_agreement", size_col],
    ascending=[True, False]
).reset_index(drop=True)

entry_df.to_csv(OUTPUT_DIR / "entry_point_communities_ranked.csv", index=False)
restricted_df.to_csv(OUTPUT_DIR / "restricted_entry_communities_ranked.csv", index=False)
ambiguous_df.to_csv(OUTPUT_DIR / "ambiguous_communities_for_review.csv", index=False)
stage3_df.to_csv(OUTPUT_DIR / "stage3_community_intelligence.csv", index=False)

print("Saved:")
print("- entry_point_communities_ranked.csv")
print("- restricted_entry_communities_ranked.csv")
print("- ambiguous_communities_for_review.csv")
print("- stage3_community_intelligence.csv")

Saved:
- entry_point_communities_ranked.csv
- restricted_entry_communities_ranked.csv
- ambiguous_communities_for_review.csv
- stage3_community_intelligence.csv


In [9]:
print("Merged labels shape:", merged_labels_df.shape)
print("Stage 3 shape:", stage3_df.shape)
print("Entry communities:", entry_df.shape)
print("Restricted communities:", restricted_df.shape)
print("Ambiguous communities:", ambiguous_df.shape)

Merged labels shape: (1307, 16)
Stage 3 shape: (1307, 62)
Entry communities: (262, 62)
Restricted communities: (120, 62)
Ambiguous communities: (131, 62)


In [10]:
print("Merged labels shape:", merged_labels_df.shape)
print("Stage 3 shape:", stage3_df.shape)
print("Entry communities:", entry_df.shape)
print("Restricted communities:", restricted_df.shape)
print("Ambiguous communities:", ambiguous_df.shape)

Merged labels shape: (1307, 16)
Stage 3 shape: (1307, 62)
Entry communities: (262, 62)
Restricted communities: (120, 62)
Ambiguous communities: (131, 62)


In [11]:
stage3_df["entry_accessibility"].value_counts(dropna=False)

,count
entry_accessibility,
moderate_entry_barrier,923
clear_entry_point,264
restricted_entry_point,120
